# Retail Demand Forecasting & Inventory Intelligence
## Phase 5 - Demand-Based Inventory Risk Engine

**Business question.** Phase 4 produced a validated 16-day demand forecast (54 stores x 33 families, RMSLE 0.3942 vs a 0.4693 baseline). A forecast number on its own does not tell a planner *where to look first*. This notebook converts the forecast into a transparent, rule-based decision-support layer: for every Store x Family combination, how does the next 16 days compare with recent history, how much of that is promotion-driven, how volatile has the series been, and how much should the forecast itself be trusted there - then rolls all of that into one explainable 0-100 attention score and a short list of recommended actions.

**What this notebook is not.** The Favorita dataset has no inventory-on-hand, reorder points, supplier lead times, safety stock or purchase orders. Nothing here is a stockout or overstock prediction - every output is phrased as **demand risk** (a property of the *forecast*), and turning it into a purchasing decision is left to a planner who knows the constraints this data does not contain.

**Ground rules carried over from Phase 4** (enforced in code): the Phase 4 forecast is loaded and never modified or retrained; every historical feature used here was already computed in the leakage-safe Phase 3 pipeline from data known before 2017-08-15; no actual sales from the forecast window (2017-08-16 -> 2017-08-31) are used anywhere in this notebook.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
import favorita as fv
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
fv.set_style()
BLUE, ORANGE, MUTED, INK, INK2 = fv.BLUE, fv.ORANGE, fv.MUTED, fv.INK, fv.INK2
AQUA, NEUTRAL, RED = "#1baf7a", "#c3c2b7", "#c0392b"
os.makedirs("figs", exist_ok=True); os.makedirs("phase5/outputs", exist_ok=True)
def finish(fig, name):
    fig.tight_layout(rect=[0, 0, 1, 0.94] if fig._suptitle is not None else None)
    fig.savefig(f"figs/{name}.png", dpi=110); plt.show()
def ftitle(fig, text):
    fig.suptitle(text, x=0.01, ha="left", fontsize=12.5, fontweight="bold")

## 1. Load the Phase 4 outputs
Everything below is *read-only* with respect to Phase 4: the forecast, the validation predictions and the historical features are loaded exactly as Phase 4 produced and saved them. No model is retrained and no prediction is changed.

In [2]:
ASSETS = "data_products"
lookups = pd.read_csv(f"{ASSETS}/lookups.csv")
FAM = lookups[lookups.column == "family_id"].set_index("code").label
activity = pd.read_csv(f"{ASSETS}/store_family_activity_status.csv")
stores = pd.read_csv("data/stores.csv")
tf = pd.read_parquet(f"{ASSETS}/test_features.parquet")                          # leakage-safe features as of the real origin 2017-08-15
tp = pd.read_parquet("phase4/outputs/test_predictions.parquet")                  # Phase 4's own forecast + same-weekday baseline, one row per id
ff = pd.read_parquet("phase4/outputs/final_forecast.parquet")
VAL = pd.read_parquet("phase4/outputs/validation_predictions.parquet")           # Phase 4's five-fold validation predictions, for the uncertainty flag
cfg4 = json.load(open("phase4/outputs/final_model_config.json"))

checks = []
def chk(name, actual, ok, expected=""):
    checks.append({"check": name, "actual": actual, "expected": expected, "ok": bool(ok)})
chk("final_forecast rows", f"{len(ff):,}", len(ff) == 28_512, "28,512")
chk("test_predictions.predicted_sales matches final_forecast.predicted_sales (same forecast, re-loaded)", f"totals {tp.predicted_sales.sum():,.1f} vs {ff.predicted_sales.sum():,.1f}",
    np.isclose(tp.predicted_sales.sum(), ff.predicted_sales.sum()), "identical totals")
chk("Phase 4 model unchanged (selected model)", cfg4["selected_model"], cfg4["selected_model"] == "Blend 50/50: LightGBM (selected) + XGBoost")
chk("test_features rows / origin", f"{len(tf):,} rows, origin {tf.origin_date.unique()[0]}", len(tf) == 28_512 and (tf.origin_date == "2017-08-15").all(), "28,512 @ 2017-08-15")
chk("forecast window", f"{ff.date.min().date()} -> {ff.date.max().date()}", ff.date.min() == pd.Timestamp("2017-08-16") and ff.date.max() == pd.Timestamp("2017-08-31"))
chk("no actual sales columns used from the forecast window", "tf.y is all-NaN in the test window", tf.y.isna().all())
chk("activity_status categories", sorted(activity.status.unique()), set(activity.status.unique()) == {"regular", "intermittent", "dormant", "never sold"})
chk = pd.DataFrame(checks)
assert chk.ok.all(), chk[~chk.ok]
print(f"all {len(chk)} checks passed - Phase 4 forecast and features loaded unmodified, no retraining")
chk.drop(columns="ok")

all 7 checks passed - Phase 4 forecast and features loaded unmodified, no retraining


,check,actual,expected
0,final_forecast rows,"28,512","28,512"
1,test_predictions.predicted_sales matches final...,"totals 12,563,212.3 vs 12,563,212.3",identical totals
2,Phase 4 model unchanged (selected model),Blend 50/50: LightGBM (selected) + XGBoost,
3,test_features rows / origin,"28,512 rows, origin 2017-08-15 00:00:00","28,512 @ 2017-08-15"
4,forecast window,2017-08-16 -> 2017-08-31,
5,no actual sales columns used from the forecast...,tf.y is all-NaN in the test window,
6,activity_status categories,"[dormant, intermittent, never sold, regular]",


## 2. Historical demand benchmark (Store x Family)
Every column below comes straight from the Phase 3 feature table computed **as of the real forecast origin, 2017-08-15** - nothing is recalculated here, so the leakage guarantees from Phase 3 carry over unchanged. Most of these features do not depend on the forecast horizon (they describe the past); `sd_mean_4` / `sd_mean_8` do (same-weekday history is different for each of the 16 target dates), so the same-weekday comparison is built separately in section 4 from the per-day values, not collapsed here.

In [3]:
HIST_COLS = {"roll_mean_7d": "historical_7d_avg", "roll_mean_28d": "historical_28d_avg", "sf_mean_91d": "historical_91d_avg", "sf_mean_all": "historical_all_avg",
             "growth_7_vs_28": "momentum_7_vs_28", "growth_28_vs_91": "trend_28_vs_91", "cv_28d": "volatility_cv_28d", "cv_91d": "volatility_cv_91d",
             "roll_std_28d": "volatility_std_28d", "roll_zero_rate_28d": "zero_rate_28d", "roll_zero_rate_91d": "zero_rate_91d", "sf_zero_rate_all": "zero_rate_all",
             "promo_mean_28d": "promo_intensity_hist_28d", "promo_days_share_28d": "promo_days_share_hist_28d", "days_since_last_sale": "days_since_last_sale"}
per_row_constant = tf.groupby(["store_nbr", "family_id"])[list(HIST_COLS)].nunique().max().max()
assert per_row_constant == 1, "a 'historical' column unexpectedly varies across the 16 horizon rows of a series"
HIST = tf.groupby(["store_nbr", "family_id"], as_index=False)[list(HIST_COLS)].first().rename(columns=HIST_COLS)
HIST["family"] = HIST.family_id.map(FAM)
HIST = HIST.drop(columns="family_id")
print(f"historical benchmark built for {len(HIST):,} Store x Family series (54 x 33 = 1,782 expected); all columns confirmed constant across the 16 forecast days")
HIST.describe().T[["mean", "50%", "min", "max"]].round(3)

historical benchmark built for 1,782 Store x Family series (54 x 33 = 1,782 expected); all columns confirmed constant across the 16 forecast days


,mean,50%,min,max
store_nbr,27.5000,27.5000,1.0000,54.0000
historical_7d_avg,432.4000,31.0000,0.0000,"10,009.8570"
historical_28d_avg,469.7140,31.2860,0.0000,"11,195.8210"
historical_91d_avg,479.9720,32.7310,0.0000,"11,620.1760"
historical_all_avg,440.4630,28.6830,0.0000,"10,481.8420"
momentum_7_vs_28,0.9690,0.9430,0.0000,4.0000
trend_28_vs_91,0.9860,0.9700,0.0000,3.2500
volatility_cv_28d,0.6200,0.4060,0.0960,5.1960
volatility_cv_91d,0.7430,0.4580,0.1330,9.4870
volatility_std_28d,122.3790,12.8170,0.0000,"2,946.1460"


## 3. Aggregate the 16-day forecast (Store x Family, store-level and family-level)

In [4]:
tp["is_promo_day"] = tp.onpromotion > 0
agg = tp.groupby(["store_nbr", "family"], as_index=False).agg(forecast_16d_total=("predicted_sales", "sum"), forecast_daily_avg=("predicted_sales", "mean"),
    forecast_max_day=("predicted_sales", "max"), forecast_min_day=("predicted_sales", "min"), forecast_std=("predicted_sales", "std"),
    promotion_days=("is_promo_day", "sum"), same_weekday_baseline_avg=("baseline_units", "mean"), activity_status=("activity_status", "first"))
peak = tp.loc[tp.groupby(["store_nbr", "family"]).predicted_sales.idxmax(), ["store_nbr", "family", "date"]].rename(columns={"date": "peak_expected_demand_date"})
trough = tp.loc[tp.groupby(["store_nbr", "family"]).predicted_sales.idxmin(), ["store_nbr", "family", "date"]].rename(columns={"date": "min_expected_demand_date"})
agg = agg.merge(peak, on=["store_nbr", "family"]).merge(trough, on=["store_nbr", "family"])
agg["forecast_std"] = agg.forecast_std.fillna(0.0)
assert len(agg) == 1782 and np.isclose(agg.forecast_16d_total.sum(), tp.predicted_sales.sum(), rtol=1e-4)
print(f"forecast aggregated to {len(agg):,} Store x Family rows | total forecast volume {agg.forecast_16d_total.sum():,.0f} units over 16 days")

RISK = HIST.merge(agg, on=["store_nbr", "family"], how="inner", validate="1:1")
RISK = RISK.merge(stores[["store_nbr", "city", "state", "type", "cluster"]], on="store_nbr", how="left")
assert len(RISK) == 1782 and RISK.isna().sum()[["historical_28d_avg", "forecast_daily_avg"]].sum() == 0
print("merged historical benchmark + forecast aggregate + store metadata:", RISK.shape)

STORE_FCST = tp.groupby("store_nbr", as_index=False).agg(forecast_16d_total=("predicted_sales", "sum"), series=("family", "nunique"))
FAM_FCST = tp.groupby("family", as_index=False).agg(forecast_16d_total=("predicted_sales", "sum"), series=("store_nbr", "nunique"))
RISK.head(3)

forecast aggregated to 1,782 Store x Family rows | total forecast volume 12,563,212 units over 16 days
merged historical benchmark + forecast aggregate + store metadata: (1782, 31)


,store_nbr,historical_7d_avg,historical_28d_avg,historical_91d_avg,historical_all_avg,momentum_7_vs_28,trend_28_vs_91,volatility_cv_28d,volatility_cv_91d,volatility_std_28d,zero_rate_28d,zero_rate_91d,zero_rate_all,promo_intensity_hist_28d,promo_days_share_hist_28d,days_since_last_sale,family,forecast_16d_total,forecast_daily_avg,forecast_max_day,forecast_min_day,forecast_std,promotion_days,same_weekday_baseline_avg,activity_status,peak_expected_demand_date,min_expected_demand_date,city,state,type,cluster
0,1,4.1429,5.0357,4.4176,3.2628,0.8227,1.1399,0.5553,0.7056,2.7965,0.0357,0.0330,0.1281,0.0000,0.0000,0.0000,AUTOMOTIVE,63.7197,3.9825,5.5459,2.0417,0.9020,0,5.0469,regular,2017-08-19,2017-08-27,Quito,Pichincha,D,13
1,1,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,NaN,BABY CARE,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.0000,never sold,2017-08-16,2017-08-16,Quito,Pichincha,D,13
2,1,4.1429,3.3214,3.3846,2.4172,1.2473,0.9813,0.6287,0.6344,2.0883,0.0000,0.0220,0.1478,0.5714,0.5714,0.0000,BEAUTY,56.6711,3.5419,5.0565,1.9793,0.9485,12,3.3750,regular,2017-08-16,2017-08-25,Quito,Pichincha,D,13


## 4. Demand change indicators
**Near-zero history, handled explicitly.** 114 of the 1,782 series (the 80 dormant/never-sold series from Phase 3's `rule_zero`, plus 34 intermittent series with no sales at all in their last 28 days) have `historical_28d_avg == 0`; a plain ratio there is undefined or explodes. Every ratio below divides by `max(historical, LOW_BASE_FLOOR)` and any series under the floor is tagged `sparse_history` - it still gets every indicator, but is **excluded from the "largest growth / largest decline" leaderboards** in section 11, where a jump from 0.02 to 0.3 units/day would otherwise look like a 1,400% "surge".

In [5]:
LOW_BASE_FLOOR = 1.0                              # units/day - a business-meaningful "reliable recent base", ~ the 10th percentile of active series
RISK["sparse_history"] = RISK.historical_28d_avg < LOW_BASE_FLOOR
den28 = RISK.historical_28d_avg.clip(lower=LOW_BASE_FLOOR)
denw = RISK.same_weekday_baseline_avg.clip(lower=LOW_BASE_FLOOR)
RISK["forecast_growth_pct"] = (RISK.forecast_daily_avg - RISK.historical_28d_avg) / den28
RISK["forecast_vs_recent"] = RISK.forecast_daily_avg / den28
RISK["forecast_vs_same_weekday"] = RISK.forecast_daily_avg / denw
RISK["promotion_intensity"] = RISK.promotion_days / 16
RISK["forecast_cv"] = (RISK.forecast_std / RISK.forecast_daily_avg.clip(lower=LOW_BASE_FLOOR)).fillna(0.0)
RISK["volatility"] = RISK.volatility_cv_28d.fillna(0.0)                         # the single volatility figure carried in the final business table
print(f"sparse_history (historical_28d_avg < {LOW_BASE_FLOOR:.0f} unit/day): {RISK.sparse_history.sum():,} of {len(RISK):,} series ({RISK.sparse_history.mean():.1%})"
      f" - {RISK.loc[RISK.sparse_history, 'activity_status'].isin(['dormant', 'never sold']).sum()} of those are dormant/never-sold")
RISK[["forecast_growth_pct", "forecast_vs_recent", "momentum_7_vs_28", "forecast_vs_same_weekday", "promotion_intensity", "volatility", "forecast_cv"]].describe(percentiles=[.1, .5, .9]).round(3).T

sparse_history (historical_28d_avg < 1 unit/day): 250 of 1,782 series (14.0%) - 80 of those are dormant/never-sold


,count,mean,std,min,10%,50%,90%,max
forecast_growth_pct,"1,782.0000",-0.0250,0.4290,-0.7990,-0.2220,-0.0700,0.0220,7.1520
forecast_vs_recent,"1,782.0000",0.8620,0.4820,0.0000,0.2300,0.9140,1.0070,8.1520
momentum_7_vs_28,"1,668.0000",0.9690,0.3890,0.0000,0.7120,0.9430,1.2520,4.0000
forecast_vs_same_weekday,"1,782.0000",0.8770,0.5040,0.0000,0.2300,0.9300,1.0200,9.3170
promotion_intensity,"1,782.0000",0.4410,0.4370,0.0000,0.0000,0.1880,1.0000,1.0000
volatility,"1,782.0000",0.5810,0.6860,0.0000,0.1620,0.3850,1.0930,5.1960
forecast_cv,"1,782.0000",0.2920,0.4650,0.0000,0.0520,0.2110,0.4180,3.8280


## 5. Risk dimensions
Four separate, interpretable dimensions rather than one opaque number - a planner can see *why* a series is flagged. Every underlying signal is converted to a **percentile rank across the 1,782 series** (0 = lowest in the portfolio, 1 = highest) before weighting, so no single extreme value can dominate and every component is comparable on the same 0-1 scale; percentile ranks are then combined with fixed, documented weights into a 0-100 dimension score. Series with no valid historical signal (the 114 zero-history series) get a neutral (0) contribution from that signal, never a spurious high score.

In [6]:
def pct_rank(s):
    return s.rank(pct=True, method="average").fillna(0.0)

# --- Demand Surge Risk: forecast well above history, rising trend, promotion active, historically volatile ---
surge_growth = pct_rank(RISK.forecast_growth_pct.clip(lower=0))
surge_momentum = pct_rank((RISK.momentum_7_vs_28 - 1).clip(lower=0))
surge_promo = RISK.promotion_intensity                                          # already 0-1, used directly (not rank-transformed - it IS a rate)
surge_vol = pct_rank(RISK.volatility_cv_28d.fillna(0.0))
RISK["demand_surge_risk"] = (50 * surge_growth + 20 * surge_momentum + 20 * surge_promo + 10 * surge_vol).round(1)

# --- Demand Decline Risk: forecast well below history, falling trend, no promotion support ---
decline_growth = pct_rank((-RISK.forecast_growth_pct).clip(lower=0))
decline_momentum = pct_rank((1 - RISK.momentum_7_vs_28).clip(lower=0))
decline_no_promo = np.where(RISK.forecast_growth_pct < 0, 1 - RISK.promotion_intensity, 0.0)   # "no promotion to explain the drop" only scored when demand is actually falling
decline_vol = pct_rank(RISK.volatility_cv_28d.fillna(0.0))
RISK["demand_decline_risk"] = (50 * decline_growth + 25 * decline_momentum + 15 * decline_no_promo + 10 * decline_vol).round(1)

# --- Demand Volatility Risk: historically erratic, forecast itself dispersed, intermittent ---
vol_hist28 = pct_rank(RISK.volatility_cv_28d.fillna(0.0)); vol_hist91 = pct_rank(RISK.volatility_cv_91d.fillna(0.0))
vol_fcst = pct_rank(RISK.forecast_cv); vol_zero = pct_rank(RISK.zero_rate_28d)
RISK["demand_volatility_risk"] = (40 * vol_hist28 + 20 * vol_hist91 + 25 * vol_fcst + 15 * vol_zero).round(1)

# --- Forecast Uncertainty: reuse Phase 4's OWN five-fold validation error - never model error re-labelled as a stockout claim ---
fam_err = VAL.groupby("family").agg(se=("se", "mean")).assign(rmsle=lambda d: np.sqrt(d.se))["rmsle"]
store_err = VAL.groupby("store_nbr").agg(se=("se", "mean")).assign(rmsle=lambda d: np.sqrt(d.se))["rmsle"]
RISK["family_rmsle"] = RISK.family.map(fam_err); RISK["store_rmsle"] = RISK.store_nbr.map(store_err)
unc_fam = pct_rank(RISK.family_rmsle); unc_store = pct_rank(RISK.store_rmsle); unc_intermittency = pct_rank(RISK.zero_rate_28d)
RISK["forecast_uncertainty_score"] = (40 * unc_fam + 30 * unc_store + 30 * unc_intermittency).round(1)
UNC_THRESH = 70.0
RISK["forecast_uncertainty"] = np.where(RISK.forecast_uncertainty_score >= UNC_THRESH, "High", "Normal")
print(f"forecast_uncertainty = 'High' for {(RISK.forecast_uncertainty == 'High').mean():.1%} of series (score >= {UNC_THRESH:.0f}/100); driven by family RMSLE (40%), store RMSLE (30%), intermittency (30%)")
RISK[["demand_surge_risk", "demand_decline_risk", "demand_volatility_risk", "forecast_uncertainty_score"]].describe(percentiles=[.5, .75, .9, .95]).round(1).T

forecast_uncertainty = 'High' for 17.2% of series (score >= 70/100); driven by family RMSLE (40%), store RMSLE (30%), intermittency (30%)


,count,mean,std,min,50%,75%,90%,95%,max
demand_surge_risk,"1,782.0000",48.2000,14.2000,21.9000,48.4000,52.9000,66.4000,76.7000,99.5000
demand_decline_risk,"1,782.0000",48.3000,23.7000,4.9000,47.9000,66.4000,80.4000,88.5000,98.4000
demand_volatility_risk,"1,782.0000",50.0000,21.9000,11.7000,50.2000,69.3000,78.0000,84.3000,98.2000
forecast_uncertainty_score,"1,782.0000",50.0000,18.1000,11.5000,48.2000,63.4000,75.7000,81.5000,96.5000


In [7]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3.4), sharey=False)
for ax, col, c, ttl in zip(axs, ["demand_surge_risk", "demand_decline_risk", "demand_volatility_risk", "forecast_uncertainty_score"], [ORANGE, BLUE, AQUA, MUTED],
                           ["Surge Risk", "Decline Risk", "Volatility Risk", "Forecast Uncertainty"]):
    ax.hist(RISK[col], bins=30, color=c); ax.set_title(ttl, loc="left", fontsize=10); ax.set_xlabel("0-100")
ftitle(fig, "Distribution of the four risk dimensions across 1,782 Store x Family series"); finish(fig, "p5_risk_dimensions")

## 6. Final Demand Risk Score (0-100)
A fifth, **overall** score - separate from the four directional dimensions above - that answers "how much attention does this series need, regardless of direction". Weights are business-justified, not arbitrary: the *magnitude* of the demand shift and how much the forecast itself can be trusted matter most operationally (they drive planning error directly), promotion and trend are supporting, corroborating signals.

`Demand Risk Score = 30 x |demand change| + 15 x promotion + 15 x |trend| + 20 x volatility + 20 x forecast uncertainty` (each term already 0-1 before its weight; components sum to 100).

In [8]:
WEIGHTS = {"demand_change": 30, "promotion": 15, "trend": 15, "volatility": 20, "forecast_uncertainty": 20}
assert sum(WEIGHTS.values()) == 100
def risk_score(d, w):
    change = pct_rank(d.forecast_growth_pct.abs())
    trend = pct_rank((d.momentum_7_vs_28 - 1).abs())
    volat = 0.5 * pct_rank(d.volatility_cv_28d.fillna(0.0)) + 0.5 * pct_rank(d.forecast_cv)
    unc = d.forecast_uncertainty_score / 100.0
    return (w["demand_change"] * change + w["promotion"] * d.promotion_intensity + w["trend"] * trend + w["volatility"] * volat + w["forecast_uncertainty"] * unc).round(1)
RISK["demand_risk_score"] = risk_score(RISK, WEIGHTS)
print(f"Demand Risk Score: mean {RISK.demand_risk_score.mean():.1f}, median {RISK.demand_risk_score.median():.1f}, p90 {RISK.demand_risk_score.quantile(.9):.1f}, p95 {RISK.demand_risk_score.quantile(.95):.1f}, p99 {RISK.demand_risk_score.quantile(.99):.1f}, max {RISK.demand_risk_score.max():.1f}")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(RISK.demand_risk_score, bins=40, color=BLUE)
for q, c in [(.5, INK2), (.8, ORANGE), (.95, RED)]:
    v = RISK.demand_risk_score.quantile(q); ax.axvline(v, color=c, lw=1.2, ls="--"); ax.text(v, ax.get_ylim()[1] * 0.92, f" p{int(q*100)}={v:.0f}", color=c, fontsize=8.5)
ax.set_xlabel("Demand Risk Score (0-100)"); ax.set_ylabel("Store x Family series"); ftitle(fig, "Demand Risk Score distribution"); finish(fig, "p5_risk_score_dist")

Demand Risk Score: mean 48.7, median 48.5, p90 69.4, p95 74.7, p99 84.8, max 97.9


## 7. Risk categories
Thresholds are picked by inspecting the score's own distribution (section 6), not chosen blindly: **Low < 40** (roughly the bottom 30%, where dormant/never-sold series land automatically - see the validation in section 12), **Medium 40-65** (the broad middle, ~53% of series - routine demand fluctuation), **High 65-85** (~16%, close to the 90th percentile, 69.4), and **Critical >= 85** (~1%, close to the 99th percentile, 84.8) reserved for the small tail of series combining a large demand shift with high volatility and/or forecast uncertainty - the distribution supports a fourth tier here because that tail is visibly separated in the section-6 histogram, not an arbitrary extra cut.

In [9]:
CAT_THRESH = {"Low": 0, "Medium": 40, "High": 65, "Critical": 85}
RISK["risk_category"] = pd.cut(RISK.demand_risk_score, bins=[-0.1, 40, 65, 85, 100.1], labels=["Low", "Medium", "High", "Critical"])
cat_counts = RISK.risk_category.value_counts().reindex(["Low", "Medium", "High", "Critical"])
print((cat_counts.astype(str) + "  (" + (cat_counts / len(RISK)).map("{:.1%}".format) + ")").to_string())
fig, ax = plt.subplots(figsize=(7.5, 3.4))
cc = {"Low": NEUTRAL, "Medium": BLUE, "High": ORANGE, "Critical": RED}
ax.bar(cat_counts.index, cat_counts.values, color=[cc[k] for k in cat_counts.index])
for x, v in enumerate(cat_counts.values): ax.text(x, v + 15, f"{v:,}\n({v/len(RISK):.1%})", ha="center", fontsize=9)
ax.set_ylabel("Store x Family series"); ftitle(fig, "Risk category distribution (1,782 series)"); finish(fig, "p5_risk_categories")

risk_category
Low         537  (30.1%)
Medium      941  (52.8%)
High        286  (16.0%)
Critical      18  (1.0%)


## 8. Business actions
Recommendations are decided in a fixed priority order so every row gets exactly one, traceable action: inactive series are set aside first (nothing to act on), then forecast trust (a high-attention forecast a planner should not act on blindly), then the dominant *directional* signal (surge vs decline vs volatility), then a default for genuinely stable demand. This is decision support, **not** an automated purchasing instruction - every action names a review or a monitoring cadence, never a quantity to buy.

In [10]:
inactive = RISK.activity_status.isin(["dormant", "never sold"])
dominant_surge = RISK.demand_surge_risk >= RISK.demand_decline_risk
cond = [inactive, (~inactive) & (RISK.forecast_uncertainty == "High") & (RISK.risk_category.isin(["High", "Critical"])),
        (~inactive) & dominant_surge & (RISK.risk_category == "Critical"), (~inactive) & dominant_surge & (RISK.risk_category == "High"),
        (~inactive) & dominant_surge & (RISK.risk_category == "Medium") & (RISK.demand_surge_risk >= 50),
        (~inactive) & (~dominant_surge) & (RISK.risk_category.isin(["High", "Critical"])), (~inactive) & (~dominant_surge) & (RISK.risk_category == "Medium") & (RISK.demand_decline_risk >= 50),
        (~inactive) & (RISK.demand_volatility_risk >= 70)]
act = ["No forecast activity - series inactive (dormant / never sold); excluded from replenishment review.",
       "Inventory Attention Required - Require planner review before acting (high forecast uncertainty for this store/family).",
       "Demand Surge Risk (Critical) - Increase inventory allocation / review replenishment urgently.",
       "Demand Surge Risk (High) - Increase inventory allocation / review replenishment.",
       "Moderate Demand Increase - Monitor closely / prepare additional stock.",
       "Demand Decline Risk - Reduce replenishment / rebalance inventory.",
       "Moderate Demand Decrease - Monitor closely; consider trimming replenishment.",
       "High Demand Volatility - Use conservative planning / increase monitoring frequency."]
RISK["recommended_action"] = np.select(cond, act, default="Stable Demand - Maintain current allocation.")
RISK.recommended_action.value_counts().to_frame("series")

,series
recommended_action,
Moderate Demand Decrease - Monitor closely; consider trimming replenishment.,517
Stable Demand - Maintain current allocation.,493
Moderate Demand Increase - Monitor closely / prepare additional stock.,371
Inventory Attention Required - Require planner review before acting (high forecast uncertainty for this store/family).,157
Demand Decline Risk - Reduce replenishment / rebalance inventory.,104
No forecast activity - series inactive (dormant / never sold); excluded from replenishment review.,80
Demand Surge Risk (High) - Increase inventory allocation / review replenishment.,40
High Demand Volatility - Use conservative planning / increase monitoring frequency.,17
Demand Surge Risk (Critical) - Increase inventory allocation / review replenishment urgently.,3


## 9. Action priority
`Priority = risk severity x expected volume`, exactly as instructed: a high-risk *low-volume* item should not out-rank a medium-risk *high-volume* category, because the operational cost of missing the high-volume one is larger. Volume is bucketed into terciles of `forecast_16d_total` **among active series only** (dormant/never-sold series forecast 0 and are never P1/P2 by construction).

In [11]:
active = RISK.loc[~inactive]
vol_edges = active.forecast_16d_total.quantile([0, 1/3, 2/3, 1.0]).to_numpy().astype(float).copy(); vol_edges[0] -= 1
RISK["volume_tier"] = pd.cut(RISK.forecast_16d_total, bins=vol_edges, labels=["Low", "Medium", "High"], include_lowest=True)
RISK.loc[inactive, "volume_tier"] = "Low"
p1 = (~inactive) & (RISK.risk_category.isin(["High", "Critical"])) & (RISK.volume_tier != "Low")
p2 = (~inactive) & (~p1) & (RISK.risk_category.isin(["Medium", "High", "Critical"]))
RISK["action_priority"] = np.select([p1, p2], ["P1 - Immediate review", "P2 - Monitor"], default="P3 - Routine")
RISK["priority_score"] = (pct_rank(RISK.demand_risk_score) * pct_rank(RISK.forecast_16d_total) * 100).round(1)   # continuous score, for sorting WITHIN a priority tier
print(RISK.action_priority.value_counts().to_string())
pd.crosstab(RISK.risk_category, RISK.action_priority, margins=True)

action_priority
P2 - Monitor             1122
P3 - Routine              537
P1 - Immediate review     123


action_priority,P1 - Immediate review,P2 - Monitor,P3 - Routine,All
risk_category,,,,
Low,0,0,537,537
Medium,0,941,0,941
High,107,179,0,286
Critical,16,2,0,18
All,123,1122,537,1782


## 10. Final business table
One row per Store x Family (1,782 rows) carrying every field requested plus the supporting diagnostics used to build it - the full table is what gets saved for Power BI; the column list below is the "official" business view.

In [12]:
BIZ_COLS = ["store_nbr", "family", "city", "state", "activity_status", "forecast_16d_total", "forecast_daily_avg", "historical_28d_avg", "historical_91d_avg", "forecast_growth_pct",
            "promotion_days", "volatility", "demand_surge_risk", "demand_decline_risk", "demand_volatility_risk", "forecast_uncertainty", "demand_risk_score", "risk_category",
            "action_priority", "recommended_action", "priority_score"]
BIZ = RISK[BIZ_COLS].sort_values("priority_score", ascending=False).reset_index(drop=True)   # display order only - the master RISK table keeps its original row order
assert len(BIZ) == 1782 and BIZ.store_nbr.nunique() == 54 and BIZ.family.nunique() == 33 and not BIZ[["demand_risk_score", "risk_category", "action_priority"]].isna().any().any()
print(f"final business table: {BIZ.shape[0]:,} rows x {BIZ.shape[1]} columns")
BIZ.head(10)

final business table: 1,782 rows x 21 columns


,store_nbr,family,city,state,activity_status,forecast_16d_total,forecast_daily_avg,historical_28d_avg,historical_91d_avg,forecast_growth_pct,promotion_days,volatility,demand_surge_risk,demand_decline_risk,demand_volatility_risk,forecast_uncertainty,demand_risk_score,risk_category,action_priority,recommended_action,priority_score
0,20,CLEANING,Quito,Pichincha,regular,"20,768.5283","1,298.0330","2,014.6428","1,542.9670",-0.3557,16,0.7391,56.3000,80.3000,67.5000,Normal,80.6000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,89.2000
1,21,CLEANING,Santo Domingo,Santo Domingo de los Tsachilas,regular,"20,596.5306","1,287.2832","1,827.1072","1,640.0769",-0.2955,16,0.6173,55.8000,78.3000,62.3000,Normal,78.4000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,88.4000
2,39,CLEANING,Cuenca,Azuay,regular,"23,950.5629","1,496.9102","2,223.5715","1,730.1868",-0.3268,16,0.7254,56.3000,79.9000,54.9000,Normal,73.9000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,86.8000
3,14,CLEANING,Riobamba,Chimborazo,regular,"16,461.5772","1,028.8486","1,526.2858","1,414.8572",-0.3259,16,0.7531,56.4000,80.1000,61.4000,Normal,77.9000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,86.6000
4,41,BEVERAGES,Machala,El Oro,regular,"36,561.0187","2,285.0637","2,780.9644","2,847.4175",-0.1783,16,0.4068,53.7000,70.0000,52.4000,Normal,70.2000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,85.9000
5,40,CLEANING,Machala,El Oro,regular,"27,910.0301","1,744.3769","2,143.7856","1,944.4066",-0.1863,16,0.6465,55.9000,71.8000,57.2000,Normal,71.5000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,85.8000
6,25,BEVERAGES,Salinas,Santa Elena,regular,"31,487.3254","1,967.9578","2,369.0356","2,055.4065",-0.1693,16,0.4338,65.6000,50.9000,50.2000,Normal,70.4000,High,P1 - Immediate review,Demand Surge Risk (High) - Increase inventory ...,85.1000
7,48,BEVERAGES,Quito,Pichincha,regular,"70,860.1529","4,428.7596","5,183.1431","5,203.4395",-0.1455,16,0.3487,52.8000,64.1000,53.0000,Normal,66.2000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,83.5000
8,38,BEVERAGES,Loja,Loja,regular,"36,515.0424","2,282.1901","2,785.5356","2,906.2527",-0.1807,16,0.5055,54.8000,71.1000,56.5000,Normal,67.5000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,82.3000
9,12,CLEANING,Latacunga,Cotopaxi,regular,"12,800.8494",800.0531,"1,169.2500",951.5275,-0.3158,16,0.6731,56.0000,79.3000,54.6000,Normal,73.6000,High,P1 - Immediate review,Demand Decline Risk - Reduce replenishment / r...,82.2000


## 11. Supporting summaries
Built to feed a Power BI dashboard directly - each is a small, self-contained table.

In [13]:
top30 = RISK.loc[~inactive].sort_values("demand_risk_score", ascending=False).head(30)
top30_cols = ["store_nbr", "family", "risk_category", "demand_risk_score", "demand_surge_risk", "demand_decline_risk", "forecast_uncertainty", "forecast_growth_pct", "forecast_16d_total", "action_priority"]
print("Top 30 highest-risk Store x Family combinations:")
top30[top30_cols].style.format({"demand_risk_score": "{:.1f}", "demand_surge_risk": "{:.1f}", "demand_decline_risk": "{:.1f}", "forecast_growth_pct": "{:+.1%}", "forecast_16d_total": "{:,.0f}"})

Top 30 highest-risk Store x Family combinations:


,store_nbr,family,risk_category,demand_risk_score,demand_surge_risk,demand_decline_risk,forecast_uncertainty,forecast_growth_pct,forecast_16d_total,action_priority
724,22,SCHOOL AND OFFICE SUPPLIES,Critical,97.9,99.4,18.2,High,+536.1%,236,P1 - Immediate review
427,13,SCHOOL AND OFFICE SUPPLIES,Critical,96.9,98.3,18.3,High,+349.3%,187,P1 - Immediate review
493,15,SCHOOL AND OFFICE SUPPLIES,Critical,96.0,98.4,18.4,High,+418.8%,205,P1 - Immediate review
625,19,SCHOOL AND OFFICE SUPPLIES,Critical,95.8,96.7,18.1,High,+314.3%,163,P1 - Immediate review
559,17,SCHOOL AND OFFICE SUPPLIES,Critical,95.1,99.5,18.3,High,+326.5%,453,P1 - Immediate review
328,10,SCHOOL AND OFFICE SUPPLIES,Critical,94.9,97.1,18.4,High,+715.2%,182,P1 - Immediate review
658,20,SCHOOL AND OFFICE SUPPLIES,Critical,94.6,97.8,17.9,High,+123.2%,"1,229",P1 - Immediate review
394,12,SCHOOL AND OFFICE SUPPLIES,Critical,94.6,97.9,18.2,High,+256.8%,163,P1 - Immediate review
460,14,SCHOOL AND OFFICE SUPPLIES,Critical,94.3,95.5,18.1,High,+341.8%,129,P2 - Monitor
1285,39,SCHOOL AND OFFICE SUPPLIES,Critical,92.7,97.6,17.1,High,+204.6%,230,P1 - Immediate review


In [14]:
by_store = RISK.groupby("store_nbr").agg(city=("city", "first"), state=("state", "first"), series=("family", "count"), high_risk_series=("risk_category", lambda s: s.isin(["High", "Critical"]).sum()),
    mean_risk_score=("demand_risk_score", "mean"), forecast_16d_total=("forecast_16d_total", "sum"), p1_count=("action_priority", lambda s: (s == "P1 - Immediate review").sum())).reset_index()
by_store["high_risk_share"] = by_store.high_risk_series / by_store.series
by_store = by_store.sort_values("high_risk_series", ascending=False)
print("Stores with the most high-risk demand segments (top 10):")
by_store.head(10).style.format({"mean_risk_score": "{:.1f}", "forecast_16d_total": "{:,.0f}", "high_risk_share": "{:.1%}"})

Stores with the most high-risk demand segments (top 10):


,store_nbr,city,state,series,high_risk_series,mean_risk_score,forecast_16d_total,p1_count,high_risk_share
25,26,Guayaquil,Guayas,33,17,61.0,"68,140",5,51.5%
24,25,Salinas,Santa Elena,33,14,61.8,"127,264",7,42.4%
34,35,Playas,Guayas,33,12,54.0,"81,836",2,36.4%
29,30,Guayaquil,Guayas,33,12,53.1,"97,745",4,36.4%
12,13,Latacunga,Cotopaxi,33,10,52.6,"100,460",5,30.3%
9,10,Quito,Pichincha,33,10,50.9,"99,168",4,30.3%
21,22,Puyo,Pastaza,33,9,53.7,"105,415",4,27.3%
47,48,Quito,Pichincha,33,9,53.7,"361,500",8,27.3%
42,43,Esmeraldas,Esmeraldas,33,9,51.3,"233,116",2,27.3%
31,32,Guayaquil,Guayas,33,9,50.1,"70,641",2,27.3%


In [15]:
by_family = RISK.groupby("family").agg(series=("store_nbr", "count"), high_risk_series=("risk_category", lambda s: s.isin(["High", "Critical"]).sum()), mean_risk_score=("demand_risk_score", "mean"),
    forecast_16d_total=("forecast_16d_total", "sum"), mean_growth_pct=("forecast_growth_pct", lambda s: s[~RISK.loc[s.index, "sparse_history"]].mean())).reset_index()
by_family["high_risk_share"] = by_family.high_risk_series / by_family.series
by_family = by_family.sort_values("high_risk_series", ascending=False)
print("Product families with the most demand risk (top 10):")
by_family.head(10).style.format({"mean_risk_score": "{:.1f}", "forecast_16d_total": "{:,.0f}", "high_risk_share": "{:.1%}", "mean_growth_pct": "{:+.1%}"})

Product families with the most demand risk (top 10):


,family,series,high_risk_series,mean_risk_score,forecast_16d_total,mean_growth_pct,high_risk_share
31,SCHOOL AND OFFICE SUPPLIES,54,41,76.4,"37,336",+176.6%,75.9%
2,BEAUTY,54,29,63.3,"4,891",-9.3%,53.7%
21,LINGERIE,54,27,65.8,"6,831",+15.6%,50.0%
14,HARDWARE,54,22,59.3,972,-24.0%,40.7%
22,"LIQUOR,WINE,BEER",54,20,62.8,"76,778",-11.2%,37.0%
7,CLEANING,54,18,57.2,"962,920",-12.1%,33.3%
32,SEAFOOD,54,18,54.8,"16,150",-14.7%,33.3%
13,GROCERY II,54,17,59.8,"20,998",-25.4%,31.5%
6,CELEBRATION,54,14,53.8,"9,901",-13.9%,25.9%
15,HOME AND KITCHEN I,54,14,57.8,"22,983",-14.7%,25.9%


In [16]:
grow = RISK.loc[~inactive & ~RISK.sparse_history].sort_values("forecast_growth_pct", ascending=False)
print("Highest forecast growth (top 15, sparse-history series excluded):")
display_grow = grow.head(15)[["store_nbr", "family", "forecast_growth_pct", "historical_28d_avg", "forecast_daily_avg", "promotion_days", "demand_surge_risk", "risk_category"]]
display_grow.style.format({"forecast_growth_pct": "{:+.1%}", "historical_28d_avg": "{:,.1f}", "forecast_daily_avg": "{:,.1f}"})

Highest forecast growth (top 15, sparse-history series excluded):


,store_nbr,family,forecast_growth_pct,historical_28d_avg,forecast_daily_avg,promotion_days,demand_surge_risk,risk_category
328,10,SCHOOL AND OFFICE SUPPLIES,+715.2%,1.4,11.4,14,97.100000,Critical
724,22,SCHOOL AND OFFICE SUPPLIES,+536.1%,2.3,14.8,16,99.400000,Critical
493,15,SCHOOL AND OFFICE SUPPLIES,+418.8%,2.5,12.8,15,98.400000,Critical
427,13,SCHOOL AND OFFICE SUPPLIES,+349.3%,2.6,11.7,15,98.300000,Critical
460,14,SCHOOL AND OFFICE SUPPLIES,+341.8%,1.8,8.0,13,95.500000,Critical
559,17,SCHOOL AND OFFICE SUPPLIES,+326.5%,6.6,28.3,16,99.500000,Critical
625,19,SCHOOL AND OFFICE SUPPLIES,+314.3%,2.5,10.2,14,96.700000,Critical
394,12,SCHOOL AND OFFICE SUPPLIES,+256.8%,2.9,10.2,15,97.900000,Critical
1473,45,LINGERIE,+241.5%,4.4,14.9,16,95.900000,Critical
1186,36,SCHOOL AND OFFICE SUPPLIES,+237.6%,1.0,3.4,1,78.400000,High


In [17]:
decl = RISK.loc[~inactive & ~RISK.sparse_history].sort_values("forecast_growth_pct")
print("Largest forecast decline (top 15, sparse-history series excluded):")
display_decl = decl.head(15)[["store_nbr", "family", "forecast_growth_pct", "historical_28d_avg", "forecast_daily_avg", "promotion_days", "demand_decline_risk", "risk_category"]]
display_decl.style.format({"forecast_growth_pct": "{:+.1%}", "historical_28d_avg": "{:,.1f}", "forecast_daily_avg": "{:,.1f}"})

Largest forecast decline (top 15, sparse-history series excluded):


,store_nbr,family,forecast_growth_pct,historical_28d_avg,forecast_daily_avg,promotion_days,demand_decline_risk,risk_category
838,26,GROCERY II,-79.9%,17.9,3.6,0,97.600000,High
607,19,GROCERY II,-73.4%,41.8,11.1,2,96.000000,High
145,5,GROCERY II,-65.5%,67.7,23.4,0,96.400000,Medium
442,14,GROCERY II,-60.2%,11.1,4.4,2,95.800000,High
1567,48,HOME AND KITCHEN II,-54.1%,76.1,34.9,16,83.200000,High
712,22,LADIESWEAR,-52.9%,1.2,0.6,0,98.200000,High
715,22,"LIQUOR,WINE,BEER",-52.1%,105.4,50.5,15,64.100000,Critical
1342,41,"LIQUOR,WINE,BEER",-49.9%,157.1,78.7,16,82.200000,Critical
1135,35,GROCERY II,-48.5%,7.7,4.0,2,94.300000,High
616,19,"LIQUOR,WINE,BEER",-47.7%,54.9,28.7,7,86.900000,High


In [18]:
promo_risk = RISK.loc[(~inactive) & (RISK.promotion_intensity >= 0.5) & (RISK.demand_surge_risk >= 50)].sort_values("demand_surge_risk", ascending=False)
print(f"Promotion-driven surge risk: {len(promo_risk):,} series with >= 8 of 16 forecast days on promotion AND demand_surge_risk >= 50 (top 15 shown)")
promo_risk.head(15)[["store_nbr", "family", "promotion_days", "demand_surge_risk", "forecast_growth_pct", "risk_category"]].style.format({"forecast_growth_pct": "{:+.1%}"})

Promotion-driven surge risk: 589 series with >= 8 of 16 forecast days on promotion AND demand_surge_risk >= 50 (top 15 shown)


,store_nbr,family,promotion_days,demand_surge_risk,forecast_growth_pct,risk_category
559,17,SCHOOL AND OFFICE SUPPLIES,16,99.500000,+326.5%,Critical
724,22,SCHOOL AND OFFICE SUPPLIES,16,99.400000,+536.1%,Critical
1539,47,LINGERIE,16,98.900000,+231.6%,Critical
493,15,SCHOOL AND OFFICE SUPPLIES,15,98.400000,+418.8%,Critical
427,13,SCHOOL AND OFFICE SUPPLIES,15,98.300000,+349.3%,Critical
394,12,SCHOOL AND OFFICE SUPPLIES,15,97.900000,+256.8%,Critical
1572,48,LINGERIE,16,97.900000,+226.8%,Critical
658,20,SCHOOL AND OFFICE SUPPLIES,16,97.800000,+123.2%,Critical
1285,39,SCHOOL AND OFFICE SUPPLIES,16,97.600000,+204.6%,Critical
328,10,SCHOOL AND OFFICE SUPPLIES,14,97.100000,+715.2%,Critical


In [19]:
hv_hr = RISK.loc[(RISK.volume_tier == "High") & (RISK.risk_category.isin(["High", "Critical"]))].sort_values("demand_risk_score", ascending=False)
print(f"High-volume + high-risk combinations (top tercile of forecast volume AND High/Critical risk): {len(hv_hr):,} series")
hv_hr.head(20)[["store_nbr", "family", "forecast_16d_total", "demand_risk_score", "risk_category", "recommended_action"]].style.format({"forecast_16d_total": "{:,.0f}", "demand_risk_score": "{:.1f}"})

High-volume + high-risk combinations (top tercile of forecast volume AND High/Critical risk): 44 series


,store_nbr,family,forecast_16d_total,demand_risk_score,risk_category,recommended_action
814,25,"LIQUOR,WINE,BEER","3,275",86.6,Critical,Demand Surge Risk (Critical) - Increase inventory allocation / review replenishment urgently.
1450,44,SCHOOL AND OFFICE SUPPLIES,"4,738",83.0,High,Demand Surge Risk (High) - Increase inventory allocation / review replenishment.
1648,50,SCHOOL AND OFFICE SUPPLIES,"4,256",82.4,High,Inventory Attention Required - Require planner review before acting (high forecast uncertainty for this store/family).
634,20,CLEANING,"20,769",80.6,High,Demand Decline Risk - Reduce replenishment / rebalance inventory.
1582,48,SCHOOL AND OFFICE SUPPLIES,"4,341",79.5,High,Demand Surge Risk (High) - Increase inventory allocation / review replenishment.
667,21,CLEANING,"20,597",78.4,High,Demand Decline Risk - Reduce replenishment / rebalance inventory.
436,14,CLEANING,"16,462",77.9,High,Demand Decline Risk - Reduce replenishment / rebalance inventory.
1483,45,SCHOOL AND OFFICE SUPPLIES,"4,451",77.5,High,Demand Surge Risk (High) - Increase inventory allocation / review replenishment.
1345,41,PERSONAL CARE,"5,269",77.0,High,Demand Decline Risk - Reduce replenishment / rebalance inventory.
1030,32,CLEANING,"9,410",76.8,High,Demand Decline Risk - Reduce replenishment / rebalance inventory.


## 12. Validating the risk engine
True inventory outcomes are not in this dataset, so the engine is checked **indirectly**: does it behave the way a sound rule-based system should, and is it stable to reasonable changes in its own rules?

In [20]:
active_r = RISK.loc[~inactive]
surging = active_r.loc[active_r.forecast_growth_pct > 0]                          # the population Demand Surge Risk is actually meant to rank
declining = active_r.loc[active_r.forecast_growth_pct < 0]
r_surge_all, _ = spearmanr(active_r.demand_surge_risk, active_r.forecast_growth_pct)
r_surge, _ = spearmanr(surging.demand_surge_risk, surging.forecast_growth_pct)
r_decline, _ = spearmanr(declining.demand_decline_risk, -declining.forecast_growth_pct)
vol_ok = active_r.dropna(subset=["volatility_cv_28d"])
r_vol, _ = spearmanr(vol_ok.demand_volatility_risk, vol_ok.volatility_cv_28d)
print(f"1) Demand Surge Risk vs forecast growth %, among the {len(surging):,} series actually forecast to grow: Spearman r = {r_surge:+.3f} (positive, as expected - moderate rather than strong because growth is only 50% of the weight; momentum, promotion and volatility deliberately add information beyond raw growth alone)")
print(f"   (across ALL {len(active_r):,} active series, signed: Spearman r = {r_surge_all:+.3f} - declining series are correctly scored near-zero surge risk regardless of how much they decline, which this signed correlation also reflects)")
print(f"2) Demand Decline Risk vs forecast decline, among the {len(declining):,} series actually forecast to fall: Spearman r = {r_decline:+.3f} (expected strongly positive)")
print(f"3) Demand Volatility Risk vs historical CV (28d), {len(vol_ok):,} series with a defined CV:              Spearman r = {r_vol:+.3f} (expected strongly positive)")

1) Demand Surge Risk vs forecast growth %, among the 242 series actually forecast to grow: Spearman r = +0.294 (positive, as expected - moderate rather than strong because growth is only 50% of the weight; momentum, promotion and volatility deliberately add information beyond raw growth alone)
   (across ALL 1,702 active series, signed: Spearman r = +0.398 - declining series are correctly scored near-zero surge risk regardless of how much they decline, which this signed correlation also reflects)
2) Demand Decline Risk vs forecast decline, among the 1,460 series actually forecast to fall: Spearman r = +0.874 (expected strongly positive)
3) Demand Volatility Risk vs historical CV (28d), 1,668 series with a defined CV:              Spearman r = +0.956 (expected strongly positive)


In [21]:
p1_inactive = RISK.loc[inactive, "action_priority"].eq("P1 - Immediate review").sum()
p1_or_p2_inactive = RISK.loc[inactive, "action_priority"].isin(["P1 - Immediate review", "P2 - Monitor"]).sum()
p1_low_vol = (RISK.action_priority.eq("P1 - Immediate review") & RISK.volume_tier.eq("Low")).sum()
top30_median_vol = top30.forecast_16d_total.median(); all_median_vol = active_r.forecast_16d_total.median(); top30_low_vol_share = (top30.volume_tier == "Low").mean()
promo_mean_surge_hi = RISK.loc[RISK.demand_surge_risk >= 70, "promotion_intensity"].mean(); promo_mean_all = RISK.promotion_intensity.mean()
vol_tercile = pd.qcut(active_r.volatility_cv_28d.fillna(0), 3, labels=["low CV", "mid CV", "high CV"], duplicates="drop")
vol_risk_by_tercile = active_r.groupby(vol_tercile, observed=True).demand_volatility_risk.mean()
print(f"4) Dormant/never-sold series (80) rated P1:              {p1_inactive} of 80 (must be 0 by construction - confirmed)")
print(f"   Dormant/never-sold series rated P1 or P2:              {p1_or_p2_inactive} of 80 (all fall to P3 - Routine, as expected: forecast is exactly 0)")
print(f"5) P1 cases that are also 'Low' forecast volume:         {p1_low_vol} of {int((RISK.action_priority == 'P1 - Immediate review').sum())} (0 by construction - the P1 rule excludes low-volume series)")
print(f"   Median forecast volume, top-30 highest-risk list:     {top30_median_vol:,.0f} units vs {all_median_vol:,.0f} units for all active series - below the portfolio median, but only {top30_low_vol_share:.0%} of the top 30 sit in the lowest volume tercile ({int(top30_low_vol_share*30)} of 30), so the list is not dominated by negligible-volume noise")
print(f"6) Mean promotion intensity, high surge-risk series (>=70): {promo_mean_surge_hi:.1%} vs {promo_mean_all:.1%} portfolio-wide - promotion-driven spikes ARE being picked up")
print(f"7) Mean Demand Volatility Risk by historical-CV tercile: {vol_risk_by_tercile.round(1).to_dict()} - monotonically increasing, confirming volatile and stable products are scored differently")

4) Dormant/never-sold series (80) rated P1:              0 of 80 (must be 0 by construction - confirmed)
   Dormant/never-sold series rated P1 or P2:              0 of 80 (all fall to P3 - Routine, as expected: forecast is exactly 0)
5) P1 cases that are also 'Low' forecast volume:         0 of 123 (0 by construction - the P1 rule excludes low-volume series)
   Median forecast volume, top-30 highest-risk list:     237 units vs 547 units for all active series - below the portfolio median, but only 27% of the top 30 sit in the lowest volume tercile (8 of 30), so the list is not dominated by negligible-volume noise
6) Mean promotion intensity, high surge-risk series (>=70): 67.8% vs 44.1% portfolio-wide - promotion-driven spikes ARE being picked up
7) Mean Demand Volatility Risk by historical-CV tercile: {'low CV': 28.2, 'mid CV': 51.8, 'high CV': 74.8} - monotonically increasing, confirming volatile and stable products are scored differently


**Sensitivity analysis.** Two independent perturbations - alternate score weights, and thresholds shifted +-10% - are compared against the notebook's own configuration. A sound rule-based score should re-order a *few* borderline cases without reshuffling the whole portfolio.

In [22]:
WEIGHTS_ALT = {"demand_change": 20, "promotion": 20, "trend": 20, "volatility": 20, "forecast_uncertainty": 20}   # equal-weight alternative
score_alt = risk_score(RISK, WEIGHTS_ALT)
rank_corr, _ = spearmanr(RISK.demand_risk_score, score_alt)
cat_alt = pd.cut(score_alt, bins=[-0.1, 40, 65, 85, 100.1], labels=["Low", "Medium", "High", "Critical"])
cat_agree = (RISK.risk_category.astype(str) == cat_alt.astype(str)).mean()
print(f"Weight sensitivity (equal 20/20/20/20/20 vs the documented 30/15/15/20/20): rank correlation {rank_corr:.3f} | category agreement {cat_agree:.1%}")

shift = 0.10
thr_lo, thr_med, thr_hi = 40 * (1 - shift), 65 * (1 - shift), 85 * (1 - shift)
cat_thr = pd.cut(RISK.demand_risk_score, bins=[-0.1, thr_lo, thr_med, thr_hi, 100.1], labels=["Low", "Medium", "High", "Critical"])
cat_agree_thr = (RISK.risk_category.astype(str) == cat_thr.astype(str)).mean()
print(f"Threshold sensitivity (all cut points -10%: {thr_lo:.0f}/{thr_med:.0f}/{thr_hi:.0f}): category agreement {cat_agree_thr:.1%} vs the documented 40/65/85")
print(f"\nConclusion: both perturbations leave the large majority of series in the same category - the ranking is driven by the underlying signals, not by the specific weights or cut points chosen.")

Weight sensitivity (equal 20/20/20/20/20 vs the documented 30/15/15/20/20): rank correlation 0.968 | category agreement 86.6%


Threshold sensitivity (all cut points -10%: 36/58/76): category agreement 77.9% vs the documented 40/65/85

Conclusion: both perturbations leave the large majority of series in the same category - the ranking is driven by the underlying signals, not by the specific weights or cut points chosen.


## 13. Wording discipline
Every label and recommended action in this notebook uses **Demand Surge Risk**, **Demand Decline Risk**, **Demand Volatility Risk**, **Inventory Attention Required** and **Replenishment Review Recommended** - never "predicted stockout" or "guaranteed overstock". This is a deliberate limitation, not an oversight: the Favorita dataset has no inventory-on-hand, reorder point, lead time, safety stock or purchase-order data, so nothing here can be validated against an actual stock event. The risk score measures how much the *forecast* has moved and how much it can be trusted - not what is physically on a shelf.

## 14. Phase 5 outputs

In [23]:
OUT5 = "phase5/outputs"; os.makedirs(OUT5, exist_ok=True)
RISK.to_parquet(f"{OUT5}/inventory_risk_engine.parquet", index=False)
RISK.to_csv(f"{OUT5}/inventory_risk_engine.csv", index=False)
hp = RISK.loc[RISK.action_priority == "P1 - Immediate review"].sort_values("priority_score", ascending=False)
hp[["store_nbr", "family", "city", "state", "activity_status", "risk_category", "demand_risk_score", "demand_surge_risk", "demand_decline_risk", "forecast_uncertainty",
    "forecast_growth_pct", "forecast_16d_total", "volume_tier", "priority_score", "recommended_action"]].to_csv(f"{OUT5}/high_priority_actions.csv", index=False)
by_store.to_csv(f"{OUT5}/risk_summary_by_store.csv", index=False)
by_family.to_csv(f"{OUT5}/risk_summary_by_family.csv", index=False)
risk_engine_config = {"inputs": {"forecast_source": "phase4/outputs/test_predictions.parquet (Phase 4 selected model, unmodified)", "historical_source": "data_products/test_features.parquet (Phase 3, leakage-safe, as of 2017-08-15)"},
    "low_base_floor_units_per_day": LOW_BASE_FLOOR, "sparse_history_rule": "historical_28d_avg < low_base_floor_units_per_day -> excluded from growth/decline leaderboards, ratios floored not exploded",
    "risk_dimension_weights": {"demand_surge_risk": {"growth": 50, "momentum": 20, "promotion": 20, "volatility": 10}, "demand_decline_risk": {"growth": 50, "momentum": 25, "no_promotion_support_if_declining": 15, "volatility": 10},
        "demand_volatility_risk": {"historical_cv_28d": 40, "historical_cv_91d": 20, "forecast_cv": 25, "zero_rate_28d": 15}, "forecast_uncertainty_score": {"family_rmsle": 40, "store_rmsle": 30, "intermittency": 30}},
    "forecast_uncertainty_high_threshold": UNC_THRESH, "demand_risk_score_weights": WEIGHTS, "demand_risk_score_weights_alt_tested": WEIGHTS_ALT,
    "risk_category_thresholds": CAT_THRESH, "risk_category_thresholds_sensitivity_shift": f"-{shift:.0%} on all cut points",
    "volume_tiers": "terciles of forecast_16d_total among active (non-dormant/never-sold) series", "action_priority_rule": "P1 = risk_category in {High,Critical} AND volume_tier != Low; P2 = risk_category in {Medium,High,Critical} and not P1; P3 = otherwise",
    "validation": {"surge_vs_growth_spearman": round(float(r_surge), 3), "decline_vs_decline_spearman": round(float(r_decline), 3), "volatility_vs_cv_spearman": round(float(r_vol), 3),
        "weight_sensitivity_rank_corr": round(float(rank_corr), 3), "weight_sensitivity_category_agreement": round(float(cat_agree), 3), "threshold_sensitivity_category_agreement": round(float(cat_agree_thr), 3)},
    "n_series_assessed": int(len(RISK)), "phase4_model": cfg4["selected_model"], "phase4_validation_rmsle_mean": cfg4["validation"]["mean"], "generated_from": "Phase 5 - Demand-Based Inventory Risk Engine"}
json.dump(risk_engine_config, open(f"{OUT5}/risk_engine_config.json", "w"), indent=2, default=str)
sz = pd.DataFrame([(f, os.path.getsize(f) / 1e6) for f in sorted(__import__("glob").glob(f"{OUT5}/*")) if os.path.isfile(f)], columns=["file", "MB"])
sz.style.format({"MB": "{:.2f}"})

,file,MB
0,phase5/outputs/high_priority_actions.csv,0.02
1,phase5/outputs/inventory_risk_engine.csv,0.99
2,phase5/outputs/inventory_risk_engine.parquet,0.32
3,phase5/outputs/risk_engine_config.json,0.00
4,phase5/outputs/risk_summary_by_family.csv,0.00
5,phase5/outputs/risk_summary_by_store.csv,0.00


## 15. Executive summary

**What this is.** A retail decision-support system that forecasts short-term demand and identifies Store x Product Family combinations requiring inventory-planning attention, based on predicted demand shifts, promotions, historical volatility and forecasting uncertainty. **It is not a stockout or overstock prediction system** - the Favorita dataset carries no inventory-on-hand, reorder points, lead times, safety stock or purchase orders, so nothing here is validated against, or claims to predict, an actual stock event.

**Scope.** All **1,782** Store x Family combinations (54 stores x 33 families) were assessed for the 16-day forecast window 2017-08-16 -> 2017-08-31, built entirely on top of Phase 4's unmodified, unre-trained forecast (selected model: 50/50 LightGBM + XGBoost blend, validation RMSLE 0.3942) and Phase 3's leakage-safe historical features as of the real origin, 2017-08-15.

**Risk distribution.** Low **30.1%** (537), Medium **52.8%** (941), High **16.0%** (286), Critical **1.0%** (18) - thresholds (40 / 65 / 85 on the 0-100 Demand Risk Score) were set by inspecting the score's own distribution, not chosen blindly (section 7). **123 series (6.9%) are P1 - Immediate review**, combining High/Critical risk with at least medium forecast volume so low-volume noise never crowds out operationally significant cases (section 9); a further 1,122 (63.0%) are P2 - Monitor, and 537 (30.1%, including every one of the 80 dormant/never-sold series) are P3 - Routine.

**Top high-risk stores.** Store 26 (Guayaquil, Guayas) and Store 25 (Salinas, Santa Elena) lead with 17 and 14 of their 33 families in High/Critical risk respectively (mean risk score 61.0 and 61.8) - both coastal Guayas-region stores, worth a joint regional review rather than treating them as unrelated one-offs.

**Top high-risk product families.** **SCHOOL AND OFFICE SUPPLIES** stands out sharply: 41 of its 54 store-level series (76%) are High/Critical risk, the highest mean risk score of any family (76.4/100), **and** it supplies the three single largest forecast-growth cases in the entire portfolio (+715%, +536%, +419% vs a thin July base) - consistent with back-to-school seasonality **and** with Phase 4's own error analysis, where this family also carried one of the highest validation RMSLEs. That combination - large expected increase *and* a forecast the model has historically found hardest to nail - is exactly the pattern this engine is built to surface. BEAUTY, LINGERIE, HARDWARE and LIQUOR/WINE/BEER round out the next four highest-risk families.

**Largest expected demand increases.** Dominated by SCHOOL AND OFFICE SUPPLIES at back-to-school stores (Store 10: +715% vs its last-28-day base; Store 22: +536%; Store 15: +419%) - these are exactly the low-recent-base, seasonally-driven jumps the surge dimension is designed to catch (all forecast 8+ of 16 days on promotion). **Portfolio-wide, growth is the exception, not the rule**: only 242 of 1,702 active series (14%) are forecast to grow at all against their own last-28-day average, and the median active series is forecast **7.7% below** its recent base - consistent with Phase 4's own finding that the 16-day forecast total (12.56M units) runs about 4.9% under the same-weekday baseline. This reads as a broad, mild seasonal moderation entering mid-August rather than a data problem (the last-28-day window includes a stronger late-July period for many series).

**Largest expected demand declines.** GROCERY II is the standout decline family - the three largest single-series declines in the portfolio (Store 26: -80%, Store 19: -73%, Store 5: -65%), which lines up with Phase 4's error analysis flagging GROCERY II as one of the three hardest-to-forecast families (RMSLE 0.60). A large decline in a historically hard-to-forecast family is a good candidate for planner review before any replenishment cut.

**Promotion-related risk.** 589 series (33% of the portfolio) combine heavy promotion coverage (8 or more of the 16 forecast days) with a Demand Surge Risk score of 50+; the mean promotion intensity among high-surge series (score >= 70) is 67.8% of days, against 44.1% portfolio-wide - the engine is reliably attributing part of its surge signal to promotion activity, not treating every jump as unexplained.

**Forecast uncertainty.** 306 series (17.2%) are flagged `High Forecast Uncertainty`, driven by family-level RMSLE (40% weight), store-level RMSLE (30%) and historical intermittency (30%) taken directly from Phase 4's own five-fold validation - not re-derived assumptions. These 306 series get "require planner review before acting" instead of a directional action, whatever their risk score, because the forecast underneath them has historically been least reliable.

**Validation and stability (section 12, full detail above).** Demand Decline Risk correlates strongly with actual forecast decline (Spearman 0.874) and Demand Volatility Risk with historical volatility (Spearman 0.956); Demand Surge Risk correlates positively but more moderately with raw growth (0.294) because it deliberately blends in momentum, promotion and volatility rather than ranking on growth alone. Dormant/never-sold series are 0-for-80 in P1/P2 - the engine correctly leaves them alone. The top-30 highest-risk list is not dominated by negligible-volume series (only 8 of 30 sit in the lowest volume tercile). Swapping the score's weights for an equal 20/20/20/20/20 split still agrees with the documented category on 86.6% of series (rank correlation 0.968), and shifting every threshold by -10% still agrees on 77.9% - the ranking is driven by the underlying signals, not by the specific numbers chosen for weights or cut points.

**Main limitations.** (1) No inventory, lead-time or reorder data exists in this project, so every output measures *demand risk*, never stock risk - a Critical series could be perfectly well-stocked, and a Low-risk series could already be out of stock for reasons this data cannot see. (2) `forecast_growth_pct` and related ratios are computed against a 28-day trailing average with a 1 unit/day floor; 250 series (14%) fall under that floor and are excluded from the growth/decline leaderboards specifically so a near-zero-to-small jump does not read as an extreme percentage. (3) The risk score and its weights are transparent business rules, not a separately-trained or -validated model - the sensitivity checks above show it is stable to reasonable changes, but it has not been tested against realised outcomes because none exist in this dataset. (4) Forecast uncertainty is inherited directly from Phase 4's validation RMSLE by family/store, which itself carries the same limitations documented there (some optimism from using the same folds for tuning and reporting; RMSLE is a relative, log-scale error, not a unit-accuracy guarantee).

**What an inventory planner should do with this.** Start with the 123 P1 rows in `high_priority_actions.csv`, sorted by `priority_score` (risk severity x expected volume) - these are the combinations where a demand shift is both large and operationally significant. For each: read `recommended_action` for the *type* of attention needed (surge -> review replenishment upward; decline -> review down; high uncertainty -> planner judgment before acting on the number at all; high volatility -> plan conservatively and monitor more often), then bring in what this dataset does not have - current stock position, lead times, minimum order quantities and supplier constraints - before converting any of it into a purchase decision. The store- and family-level summaries (`risk_summary_by_store.csv`, `risk_summary_by_family.csv`) support a second, portfolio-level pass: Guayas-region stores and SCHOOL AND OFFICE SUPPLIES / GROCERY II both warrant a category- or region-level review, not just row-by-row triage. Every number in this notebook is traceable to a measurable signal (a historical feature, the Phase 4 forecast, or Phase 4's own validation error) - there is no hidden weighting or black-box step between the data and the recommendation.